# 05 – Anomaly generation
Point / contextual / collective anomalies with ground truth. The original table is never modified.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
root = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "src").exists())
sys.path.insert(0, str(root))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
try:
    from IPython.display import display, Image
except ImportError:
    display = print
PARSED = root / "data" / "processed" / "gas_sensor_drift_parsed.csv"
USE_STANDIN = False   # True ONLY to test the notebook without UCI data (results then are NOT valid)
if USE_STANDIN:
    from src.synthetic import make_standin
    df = make_standin(5000)
    print("!! STAND-IN DATA - not UCI")
else:
    assert PARSED.exists(), "Run notebook 01 (or `python -m src.data_loader`) first"
    df = pd.read_csv(PARSED)
FIG = root / "results" / "figures"; TAB = root / "results" / "tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
print(df.shape)

In [ ]:
from src.anomaly_generation import *
from src.preprocessing import feature_columns, select_dimensions
from src import visualization as viz
from src.preprocessing import Preprocessor
cols = select_dimensions(feature_columns(df), 64)
sample = df.sample(2000, random_state=0).sort_index(); pool = df.drop(sample.index)
out = {}
for t in ['point','contextual','collective']:
    cfg = InjectionConfig(anomaly_type=t, ratio=0.05, seed=0)
    out[t] = inject_into_dataframe(sample, cols, cfg, pool)
    print(t, out[t].anomaly_type.value_counts().to_dict())
print('original sample untouched:', np.array_equal(sample[cols].to_numpy(), df.loc[sample.index, cols].to_numpy()))

In [ ]:
inj = inject_anomalies(sample[cols].to_numpy(), InjectionConfig(anomaly_type='mixed', ratio=0.1, seed=1), sample.Class.values, sample.Concentration.values, pool[cols].to_numpy(), pool.Class.values, pool.Concentration.values)
display(inj.log.head(10)); display(inj.log.type.value_counts())

## Strength sweep for point anomalies (1σ–4σ): how detectable is each level?

In [ ]:
from src.registry import make_detector; from src.evaluation import evaluate_scores
rows=[]
for s in [1,2,3,4]:
    inj_s = inject_anomalies(sample[cols].to_numpy(), InjectionConfig(anomaly_type='point', ratio=0.05, strength=float(s), seed=0))
    Z = Preprocessor('median','standard').fit_transform(inj_s.X)
    for a in ['Mahalanobis','LOF','IsolationForest']:
        rows.append(dict(strength_sigma=s, algorithm=a, roc_auc=evaluate_scores(inj_s.y, make_detector(a).fit_score(Z), .05)['roc_auc']))
display(pd.DataFrame(rows).pivot(index='algorithm', columns='strength_sigma', values='roc_auc').round(3))

In [ ]:
Z = Preprocessor('median','standard').fit_transform(out['point'][cols].to_numpy())
viz.plot_embedding(Z, out['point'].anomaly_label.values, out['point'].anomaly_type.values, None, 'pca', FIG/'nb05_point_pca.png');
out['point'].to_csv(root/'data'/'synthetic'/'example_point_anomalies.csv', index=False)

Document in the report: mechanisms (zscore / multiplicative / extreme_replace), fixed number of altered features k=4 (dilution in high-d), contextual = donor vector from another gas, collective = identical coordinated shift over a window of 10 consecutive rows.

### Result log (fill in after running)
*Not yet executed.* Copy the numbers printed above into your report; do not quote numbers from memory.